# Control de un player de vídeo a través de gestos

## Paquetes necesarios

In [1]:
import cv2
import mediapipe as mp

![Hand landmarks.]("C:/Users/34611/OneDrive/Escritorio/GTDM/Tercero/Imagen/Proyecto/Control-de-player-via-gestos/images/hand_landmarks.png")

In [2]:
def get_finger_states(hand_landmarks):
    fingers = []

    # Pulgar (compara X en lugar de Y porque el pulgar se mueve lateralmente)
    if hand_landmarks.landmark[4].x < hand_landmarks.landmark[3].x:
        fingers.append(1)  # Extendido
    else:
        fingers.append(0)  # Doblado

    # Resto de los dedos: Índice, Medio, Anular, Meñique
    tips = [8, 12, 16, 20]
    pips = [6, 10, 14, 18]

    for tip, pip in zip(tips, pips):
        if hand_landmarks.landmark[tip].y < hand_landmarks.landmark[pip].y:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers  # [pulgar, índice, medio, anular, meñique]

In [3]:
# Detecta gesto: Mano abierta o puño
def detect_custom_gesture(fingers):
    # fingers = [pulgar, índice, medio, anular, meñique]
    if fingers == [1, 1, 0, 0, 0]:
        return "L"
    elif fingers == [0, 1, 1, 0, 0]:
        return "V"
    elif fingers == [0, 1, 0, 0, 1]:
        return "Rock"
    elif fingers == [0, 0, 0, 0, 0]:
        return "Puño"
    elif fingers == [1, 1, 1, 1, 1]:
        return "Palma"
    else:
        return "Unknown"

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

# Configurar Selenium (modo visible o headless)
options = Options()
# options.add_argument('--headless')  # descomenta si quieres que sea invisible
options.add_argument("--autoplay-policy=no-user-gesture-required")
driver = webdriver.Chrome(options=options)
driver.get("https://areyousure.wavedash.com/player.html?content=Video1/glass_half.mpd")

from selenium.webdriver.support.ui import WebDriverWait
ready = driver.execute_script("return video.readyState;")
print("Video readyState:", ready) 
#WebDriverWait(driver, 10).until(
#    lambda d: d.execute_script("return video.readyState;") == 4
#)

Video readyState: 0


In [18]:
# Captura
import time
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
cap = cv2.VideoCapture(1)  #CAMBIAR A 0 SI NO VA
pos_anterior = None
currentMilis = 0
previousMilis = 0

with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print('No se puede leer la cámara')
            break

        try:
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = hands.process(image)
    
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    punta_indice = hand_landmarks.landmark[8]
                    currentMilis = time.time()
                    if pos_anterior != None:
                        if punta_indice.x-pos_anterior[8].x > 0.06 and currentMilis-previousMilis>0.5:
                            print('adelantar')
                            mov = 'der-izq'
                            driver.execute_script("video.currentTime += 5;")
                        if punta_indice.y-pos_anterior[8].y > 0.06:
                            print('subir volumen')
                            mov = 'abajo-arriba'
                            driver.execute_script("if (video.volume <= 0.9){video.volume += 0.1}else {video.volume=1};")
                        cv2.putText(image, f'Gesture: {mov}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                        previousMilis = currentMilis
                    pos_anterior = hand_landmarks.landmark

                    #fingers = get_finger_states(hand_landmarks)
                    #gesture = detect_custom_gesture(fingers)
                    #cv2.putText(image, f'Gesture: {gesture}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                    
                    
                    
                    #if gesture == 'Palma':
                     #   driver.execute_script("video.pause();")
    
            cv2.imshow('Hand Gesture Recognition', image)
        except Exception as e:
            import traceback
            print("Ocurrió un error:")
            traceback.print_exc()
            
        
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

subir volumen
subir volumen
subir volumen
subir volumen
subir volumen
subir volumen
subir volumen
subir volumen
subir volumen
subir volumen


In [17]:
# Reproducir
driver.execute_script("video.play();")

Video readyState: 0
